In [129]:
import numpy as np
import pandas as pd
import math

from matplotlib import pyplot as plt

from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn import metrics
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV
from sklearn.model_selection import TimeSeriesSplit

In [130]:
def rmse(y_true, y_pred): 
    return math.sqrt(((y_true - y_pred)**2).mean())

In [131]:
def print_score(m, X_train, y_train, X_valid, y_valid):
    print('RMSE on train set: {:.4f}'.format(rmse(m.predict(X_train), y_train)))
    print('RMSE on valid set: {:.4f}'.format(rmse(m.predict(X_valid), y_valid)))
    print('R^2 on train set: {:.4f}'.format(m.score(X_train, y_train)))
    print('R^2 on valid set: {:.4f}'.format(m.score(X_valid, y_valid)))
    if hasattr(m, 'oob_score_'): print('R^2 on oob set: {:.4f}'.format(m.oob_score_))
    return

### What we need to make this work
Okay so we need to have a df converted to a feather_file with such 
- all the features( parameters ) that we can get
- the y value we want to predict  

In [132]:
df_treated = pd.read_feather('data/treated.feather')
df_treated

Price               Open                                                 \
Ticker           BTC-USD  crude_oil   US_dollar         gold     silver   
Date                                                                      
2018-01-01  14112.200195        NaN         NaN          NaN        NaN   
2018-01-02  13625.000000  60.200001   92.239998  1302.300049  17.155001   
2018-01-03  14978.200195  60.389999   91.820000  1320.000000  17.080000   
2018-01-04  15270.700195  61.959999   92.220001  1319.400024  17.084999   
2018-01-05  15477.200195  61.900002   91.870003  1320.300049  17.205000   
...                  ...        ...         ...          ...        ...   
2026-10-07  85554.796875  89.959999  101.889999  4195.000000  60.880001   
2026-10-08  83281.468750  89.000000  102.230003  4139.899902  59.063999   
2026-10-09  81676.335938  91.290001  102.129997  4159.700195  60.669998   
2026-10-10  82546.320312        NaN         NaN          NaN        NaN   
2026-10-11  82912.226562        NaN         NaN          NaN        NaN   

Price                                                                \
Ticker       ^FVX 3month_treasury_yield         ^IXIC    Nasdaq-100   
Date                                                                  
2018-01-01    NaN                   NaN           NaN           NaN   
2018-01-02  2.228                 1.388   6937.649902   6431.589844   
2018-01-03  2.242                 1.388   7017.069824   6520.029785   
2018-01-04  2.277                 1.370   7089.500000   6595.750000   
2018-01-05  2.280                 1.370   7105.740234   6613.129883   
...           ...                   ...           ...           ...   
2026-10-07  5.072                 4.037  27430.539062  30976.250000   
2026-10-08  5.070                 4.043  27416.029297  30985.230469   
2026-10-09  4.996                 4.045  27342.269531  30900.060547   
2026-10-10    NaN                   NaN           NaN           NaN   
2026-10-11    NaN                   NaN           NaN           NaN   

Price                          ...      Close                               \
Ticker     10y_treasury_yield  ...     silver   ^FVX 3month_treasury_yield   
Date                           ...                                           
2018-01-01                NaN  ...        NaN    NaN                   NaN   
2018-01-02              2.433  ...  17.121000  2.250                 1.378   
2018-01-03              2.451  ...  17.184999  2.247                 1.370   
2018-01-04              2.473  ...  17.188999  2.268                 1.370   
2018-01-05              2.465  ...  17.205000  2.285                 1.370   
...                       ...  ...        ...    ...                   ...   
2026-10-07              5.341  ...  59.898998  5.021                 4.037   
2026-10-08              5.324  ...  59.063999  4.991                 4.043   
2026-10-09              5.236  ...  60.669998  5.021                 4.057   
2026-10-10                NaN  ...        NaN    NaN                   NaN   
2026-10-11                NaN  ...        NaN    NaN                   NaN   

Price                                                                        \
Ticker             ^IXIC    Nasdaq-100 10y_treasury_yield   ^TYX Volatility   
Date                                                                          
2018-01-01           NaN           NaN                NaN    NaN        NaN   
2018-01-02   7006.899902   6511.339844              2.465  2.811       9.77   
2018-01-03   7065.529785   6575.799805              2.447  2.785       9.15   
2018-01-04   7077.910156   6584.580078              2.453  2.786       9.22   
2018-01-05   7136.560059   6653.290039              2.476  2.811       9.22   
...                  ...           ...                ...    ...        ...   
2026-10-07  27538.689453  31160.080078              5.277  5.661      15.08   
2026-10-08  27193.339844  30725.810547              5.231  5.606      1

In [133]:
df = df_treated.iloc[:,:26]
y = df_treated["up/down"]

# time series split 

In [134]:
tscv = TimeSeriesSplit(n_splits=3)

for train_index, test_index in tscv.split(df):
    X_train , X_test  = df.iloc[train_index] , df.iloc[test_index] 
    Y_train , Y_test   = y.iloc[train_index] , y.iloc[test_index]

In [135]:
'''
for train_index, val_index in tscv.split(df):
    X_train, X_val = df.iloc[train_index], df.iloc[val_index]
    Y_train, Y_val = y.iloc[train_index], y.iloc[val_index]'''

'\nfor train_index, val_index in tscv.split(df):\n    X_train, X_val = df.iloc[train_index], df.iloc[val_index]\n    Y_train, Y_val = y.iloc[train_index], y.iloc[val_index]'

In [136]:
model = RandomForestRegressor(n_estimators= 10, n_jobs=-1, oob_score=True)

%time model.fit(X_train, Y_train)

CPU times: user 225 ms, sys: 17.8 ms, total: 243 ms
Wall time: 213 ms


/opt/anaconda3/envs/NASDAQ_Prediction-Project/lib/python3.14/site-packages/sklearn/ensemble/_forest.py:589: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.r2_score` is used.Provide a callable with signature `metric(y_true, y_pred)` to use acustom metric. Only available if `bootstrap=True`.For an illustration of out-of-bag (OOB) error estimation, see the example:ref:`sphx_glr_auto_examples_ensemble_plot_ensemble_oob.py`.",True
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requir

In [137]:
print_score(model, X_train , Y_train, X_test,Y_test)

RMSE on train set: 0.1589
RMSE on valid set: 0.4154
R^2 on train set: 0.8917
R^2 on valid set: 0.2615
R^2 on oob set: 0.2633


In [138]:
model.feature_importances_

array([0.02098406, 0.07724757, 0.01601055, 0.04682904, 0.01733029,
       0.01371241, 0.01298119, 0.02729548, 0.01385906, 0.01179826,
       0.01152426, 0.09332382, 0.08511927, 0.01569545, 0.13005595,
       0.04787992, 0.04929041, 0.01835907, 0.01907452, 0.0109334 ,
       0.01939897, 0.04657064, 0.01219832, 0.0118493 , 0.08006721,
       0.09061159])

In [139]:
model.score(X_test,Y_test)

0.2615435574472629